<a href="https://colab.research.google.com/github/zarqa-abid/ensemble-iomt-disease-prediction/blob/main/conference_ppr_(ann%2Csvm%2Cknn%2Cvoting_classifier).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ***NOW INTEGRATE ANN, SVM , KNN  IN VOTING CLASSIFIER ***

In [ ]:
!pip install scikeras[tensorflow]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 61.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2


In [ ]:
# Ensure necessary libraries are installed
!pip install pandas scikit-learn matplotlib seaborn tensorflow scikeras[tensorflow]

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# Import TensorFlow and scikeras for the ANN model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Step 1: Load the dataset
data = pd.read_csv("/content/heart_cleveland_upload.csv")

# Step 2: Preprocess the dataset
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define ANN
def create_ann(optimizer='adam', neurons=16):
    ann = Sequential()
    ann.add(Dense(neurons, input_dim=X_train.shape[1], activation='relu'))
    ann.add(Dense(4, activation='relu'))
    ann.add(Dense(1, activation='sigmoid'))
    ann.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return ann

# Create KerasClassifier for ANN
ann = KerasClassifier(build_fn=create_ann, epochs=300, batch_size=3, verbose=0)

# Hyperparameter tuning using GridSearchCV
ann_param_grid = {
    'optimizer': ['adam', 'rmsprop'],
    'neurons': [16, 32]
}
svm_param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}
knn_param_grid = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance']
}

# GridSearch for ANN
ann_grid = GridSearchCV(estimator=ann, param_grid=ann_param_grid, cv=3)
ann_grid.fit(X_train, y_train)
best_ann = ann_grid.best_estimator_

# GridSearch for SVM
svm_grid = GridSearchCV(estimator=SVC(probability=True), param_grid=svm_param_grid, cv=3)
svm_grid.fit(X_train, y_train)
best_svm = svm_grid.best_estimator_

# GridSearch for KNN
knn_grid = GridSearchCV(estimator=KNeighborsClassifier(), param_grid=knn_param_grid, cv=3)
knn_grid.fit(X_train, y_train)
best_knn = knn_grid.best_estimator_

# Fit the models
best_svm.fit(X_train, y_train)
best_knn.fit(X_train, y_train)
best_ann.fit(X_train, y_train)

# Define and train the voting classifier including ANN
voting_clf = VotingClassifier(estimators=[('svm', best_svm), ('knn', best_knn), ('ann', best_ann)], voting='soft')
voting_clf.fit(X_train, y_train)

# Step 4: Sample 30 rows from the original dataset
sample_data = data.sample(n=30, random_state=42)
X_sample = sample_data.iloc[:, :-1]
y_sample = sample_data.iloc[:, -1]

# Standardize the sample data
X_sample_scaled = scaler.transform(X_sample)

# Predict the condition values using each model
svm_sample_pred = best_svm.predict(X_sample_scaled)
knn_sample_pred = best_knn.predict(X_sample_scaled)
voting_sample_pred = voting_clf.predict(X_sample_scaled)
ann_sample_pred = (best_ann.predict(X_sample_scaled) > 0.5).astype("int32").flatten()

# Calculate and print the accuracy of each model on the sample data
svm_sample_accuracy = accuracy_score(y_sample, svm_sample_pred)
knn_sample_accuracy = accuracy_score(y_sample, knn_sample_pred)
ann_sample_accuracy = accuracy_score(y_sample, ann_sample_pred)
voting_sample_accuracy = accuracy_score(y_sample, voting_sample_pred)

print("\nAccuracy on sample data:")
print(f"SVM Accuracy: {svm_sample_accuracy:.2f}")
print(f"K-Nearest Neighbors Accuracy: {knn_sample_accuracy:.2f}")
print(f"ANN Accuracy: {ann_sample_accuracy:.2f}")
print(f"Voting Classifier Accuracy: {voting_sample_accuracy:.2f}")

# Calculate and print the other metrics of each model on the sample data
svm_sample_precision = precision_score(y_sample, svm_sample_pred)
knn_sample_precision = precision_score(y_sample, knn_sample_pred)
ann_sample_precision = precision_score(y_sample, ann_sample_pred)
voting_sample_precision = precision_score(y_sample, voting_sample_pred)

svm_sample_recall = recall_score(y_sample, svm_sample_pred)
knn_sample_recall = recall_score(y_sample, knn_sample_pred)
ann_sample_recall = recall_score(y_sample, ann_sample_pred)
voting_sample_recall = recall_score(y_sample, voting_sample_pred)

svm_sample_f1 = f1_score(y_sample, svm_sample_pred)
knn_sample_f1 = f1_score(y_sample, knn_sample_pred)
ann_sample_f1 = f1_score(y_sample, ann_sample_pred)
voting_sample_f1 = f1_score(y_sample, voting_sample_pred)

print("\nPrecision on sample data:")
print(f"SVM Precision: {svm_sample_precision:.2f}")
print(f"K-Nearest Neighbors Precision: {knn_sample_precision:.2f}")
print(f"ANN Precision: {ann_sample_precision:.2f}")
print(f"Voting Classifier Precision: {voting_sample_precision:.2f}")

print("\nRecall on sample data:")
print(f"SVM Recall: {svm_sample_recall:.2f}")
print(f"K-Nearest Neighbors Recall: {knn_sample_recall:.2f}")
print(f"ANN Recall: {ann_sample_recall:.2f}")
print(f"Voting Classifier Recall: {voting_sample_recall:.2f}")

print("\nF1 Score on sample data:")
print(f"SVM F1 Score: {svm_sample_f1:.2f}")
print(f"K-Nearest Neighbors F1 Score: {knn_sample_f1:.2f}")
print(f"ANN F1 Score: {ann_sample_f1:.2f}")
print(f"Voting Classifier F1 Score: {voting_sample_f1:.2f}")

# Plotting the accuracies
models = ['ANN', 'KNN', 'SVM', 'Ensemble']
accuracies = [ann_sample_accuracy, knn_sample_accuracy, svm_sample_accuracy, voting_sample_accuracy]

plt.figure(figsize=(10, 6))
plt.bar(models, accuracies, color=['blue', 'green', 'red', 'purple'])
plt.title('Model Accuracies')
plt.xlabel('Models')
plt.ylabel('Accuracy')

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center', va='bottom')

plt.show()
